<a href="https://colab.research.google.com/github/elpanadero321/26426/blob/main/Copia_de_Te_damos_la_bienvenida_a_Colaboratory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

extra_paths = [
    "/usr/lib/python3/dist-packages",
    "/usr/local/lib/python3.13/dist-packages",
    "/usr/local/lib/python3.12/dist-packages",
    "/usr/local/lib/python3.11/dist-packages",
]

for p in extra_paths:
    if Path(p).exists() and p not in sys.path:
        sys.path.append(p)

import json
from dataclasses import asdict, dataclass

import numpy as np
from scipy.integrate import cumulative_trapezoid, solve_ivp
from scipy.optimize import minimize

try:
    import matplotlib.pyplot as plt
    HAS_MATPLOTLIB = True
except Exception:
    HAS_MATPLOTLIB = False

C_LIGHT = 299792.458

BAO_BLOCKS = [
    {"z": 0.295, "kind": "DV", "mean": 7.93, "sigma": 0.15},
    {"z": 0.510, "kind": "DM_DH", "mean": [13.62, 20.98], "sigma": [0.25, 0.61], "rho": -0.445},
    {"z": 0.706, "kind": "DM_DH", "mean": [16.85, 20.08], "sigma": [0.32, 0.60], "rho": -0.420},
    {"z": 0.930, "kind": "DM_DH", "mean": [21.71, 17.88], "sigma": [0.28, 0.35], "rho": -0.389},
    {"z": 1.317, "kind": "DM_DH", "mean": [27.79, 13.82], "sigma": [0.69, 0.42], "rho": -0.444},
    {"z": 1.491, "kind": "DV", "mean": 26.07, "sigma": 0.67},
    {"z": 2.330, "kind": "DM_DH", "mean": [39.71, 8.52], "sigma": [0.94, 0.17], "rho": -0.477},
]

HZ_CC = np.array([
    [0.07, 69.0, 19.6], [0.09, 69.0, 12.0], [0.12, 68.6, 26.2],
    [0.17, 83.0, 8.0], [0.1791, 78.0, 6.2], [0.1993, 78.0, 6.9],
    [0.20, 72.9, 29.6], [0.27, 77.0, 14.0], [0.28, 88.8, 36.6],
    [0.3519, 85.5, 15.7], [0.3802, 86.2, 14.6], [0.40, 95.0, 17.0],
    [0.4004, 79.9, 11.4], [0.4247, 90.4, 12.8], [0.4497, 96.3, 14.4],
    [0.47, 89.0, 49.6], [0.4783, 83.8, 10.2], [0.48, 97.0, 62.0],
    [0.5929, 107.0, 15.5], [0.6797, 95.0, 10.5], [0.75, 98.8, 33.6],
    [0.7812, 96.5, 12.5], [0.8754, 124.5, 17.4], [0.88, 90.0, 40.0],
    [0.90, 117.0, 23.0], [1.037, 133.5, 17.6], [1.30, 168.0, 17.0],
    [1.363, 160.0, 33.8], [1.43, 177.0, 18.0], [1.53, 140.0, 14.0],
    [1.75, 202.0, 40.0], [1.965, 186.5, 50.6],
], dtype=float)

PLANCK_H0 = 67.4
PLANCK_SIGMA = 0.5
SHOES_H0 = 73.0
SHOES_SIGMA = 1.0
BRIDGE_PIVOT_Z = 2.33


@dataclass(frozen=True)
class FitConfig:
    omega_r0: float = 9e-5
    beta: float = 1e-4
    dvch_init: tuple[float, float, float, float] = (71.0, 0.30, 0.18, 147.0)
    lcdm_init: tuple[float, float, float] = (69.0, 0.30, 147.0)
    maxiter: int = 12000
    xatol: float = 1e-6
    fatol: float = 1e-6
    bridge_pivot_z: float = BRIDGE_PIVOT_Z


@dataclass
class FitSummary:
    model: str
    best_fit: list[float]
    chi2_total: float
    chi2_bao: float
    chi2_cc: float
    chi2_h0_pair: float
    chi2_planck_anchor: float
    chi2_shoes_anchor: float
    sigma_pair: float
    aic: float
    bic: float
    bridge_highz: float
    bridge_lowz: float


@dataclass
class JointFitResults:
    assumptions: dict
    n_data: int
    dvch: FitSummary
    lcdm: FitSummary
    delta_chi2: float
    delta_aic: float
    delta_bic: float


def rhs_dvch(z, y, n, beta):
    omega_m, omega_l, omega_r = y
    omega_m = max(omega_m, 1e-15)
    omega_l = max(omega_l, 0.0)
    omega_r = max(omega_r, 0.0)

    e2 = max(omega_m + omega_l + omega_r, 1e-20)
    e = np.sqrt(e2)

    qtilde = -e * omega_l / (1.0 + n * omega_l / omega_m) * (
        n - (beta / (1.0 + beta * e2)) * ((4.0 * omega_r + 3.0 * omega_m) / 3.0)
    )

    domega_m_dz = 3.0 * (omega_m + qtilde / e) / (1.0 + z)
    domega_l_dz = -3.0 * qtilde / (e * (1.0 + z))
    domega_r_dz = 4.0 * omega_r / (1.0 + z)
    return [domega_m_dz, domega_l_dz, domega_r_dz]


def solve_dvch_grid(zmax, H0, omega_m0, omega_r0, n, beta, npts=4500):
    omega_l0 = 1.0 - omega_m0 - omega_r0
    if omega_l0 <= 0.0:
        raise ValueError("Omega_Lambda0 must be positive.")

    z_grid = np.linspace(0.0, zmax, npts)
    solution = solve_ivp(
        lambda z, y: rhs_dvch(z, y, n=n, beta=beta),
        (0.0, zmax),
        [omega_m0, omega_l0, omega_r0],
        t_eval=z_grid,
        rtol=1e-9,
        atol=1e-12,
        method="RK45",
    )
    if not solution.success:
        raise RuntimeError(solution.message)

    omega_m, omega_l, omega_r = solution.y
    e_grid = np.sqrt(np.maximum(omega_m + omega_l + omega_r, 1e-20))
    return z_grid, e_grid


def solve_lcdm_grid(zmax, omega_m0, omega_r0, npts=4500):
    z_grid = np.linspace(0.0, zmax, npts)
    omega_l0 = 1.0 - omega_m0 - omega_r0
    e_grid = np.sqrt(
        omega_m0 * (1.0 + z_grid) ** 3
        + omega_r0 * (1.0 + z_grid) ** 4
        + omega_l0
    )
    return z_grid, e_grid


def distance_grids(z_grid, e_grid, H0):
    chi_grid = (C_LIGHT / H0) * cumulative_trapezoid(1.0 / e_grid, z_grid, initial=0.0)
    dh_grid = C_LIGHT / (H0 * e_grid)
    return chi_grid, dh_grid


def bridge_h0_curve(z_grid, e_grid, H0):
    return H0 * e_grid / (1.0 + z_grid)


def bao_chi2(z_grid, e_grid, H0, rd):
    chi_grid, dh_grid = distance_grids(z_grid, e_grid, H0)
    total = 0.0

    for block in BAO_BLOCKS:
        z = block["z"]
        dm = np.interp(z, z_grid, chi_grid)
        dh = np.interp(z, z_grid, dh_grid)

        if block["kind"] == "DV":
            dv = (z * dm * dm * dh) ** (1.0 / 3.0)
            model = dv / rd
            total += ((model - block["mean"]) / block["sigma"]) ** 2
        else:
            model = np.array([dm / rd, dh / rd])
            mean = np.array(block["mean"])
            sigma1, sigma2 = block["sigma"]
            rho = block["rho"]
            covariance = np.array([
                [sigma1 * sigma1, rho * sigma1 * sigma2],
                [rho * sigma1 * sigma2, sigma2 * sigma2],
            ])
            delta = model - mean
            total += float(delta @ np.linalg.inv(covariance) @ delta)

    return float(total)


def chronometer_chi2(z_grid, e_grid, H0):
    z = HZ_CC[:, 0]
    h_obs = HZ_CC[:, 1]
    h_err = HZ_CC[:, 2]
    h_model = H0 * np.interp(z, z_grid, e_grid)
    return float(np.sum(((h_obs - h_model) / h_err) ** 2))


def h0_anchor_chi2(z_grid, e_grid, H0, bridge_pivot_z):
    bridge = bridge_h0_curve(z_grid, e_grid, H0)
    highz_value = float(np.interp(bridge_pivot_z, z_grid, bridge))
    lowz_value = float(np.interp(0.0, z_grid, bridge))

    chi2_planck = ((highz_value - PLANCK_H0) / PLANCK_SIGMA) ** 2
    chi2_shoes = ((lowz_value - SHOES_H0) / SHOES_SIGMA) ** 2
    chi2_pair = float(chi2_planck + chi2_shoes)
    sigma_pair = float(np.sqrt(chi2_pair))

    return {
        "bridge_highz": highz_value,
        "bridge_lowz": lowz_value,
        "chi2_planck_anchor": float(chi2_planck),
        "chi2_shoes_anchor": float(chi2_shoes),
        "chi2_h0_pair": chi2_pair,
        "sigma_pair": sigma_pair,
    }


def information_criteria(chi2, n_params, n_data):
    aic = chi2 + 2 * n_params
    bic = chi2 + n_params * np.log(n_data)
    return float(aic), float(bic)


def build_summary(model_name, best_fit, z_grid, e_grid, H0, rd, n_params, n_data, bridge_pivot_z):
    chi2_bao = bao_chi2(z_grid, e_grid, H0, rd)
    chi2_cc = chronometer_chi2(z_grid, e_grid, H0)
    anchor = h0_anchor_chi2(z_grid, e_grid, H0, bridge_pivot_z)
    chi2_total = chi2_bao + chi2_cc + anchor["chi2_h0_pair"]
    aic, bic = information_criteria(chi2_total, n_params=n_params, n_data=n_data)

    return FitSummary(
        model=model_name,
        best_fit=list(map(float, best_fit)),
        chi2_total=float(chi2_total),
        chi2_bao=float(chi2_bao),
        chi2_cc=float(chi2_cc),
        chi2_h0_pair=anchor["chi2_h0_pair"],
        chi2_planck_anchor=anchor["chi2_planck_anchor"],
        chi2_shoes_anchor=anchor["chi2_shoes_anchor"],
        sigma_pair=anchor["sigma_pair"],
        aic=aic,
        bic=bic,
        bridge_highz=anchor["bridge_highz"],
        bridge_lowz=anchor["bridge_lowz"],
    )


def run_joint_fit(config=FitConfig()):
    zmax = max(np.max(HZ_CC[:, 0]), max(block["z"] for block in BAO_BLOCKS))
    n_data = len(HZ_CC) + sum(1 if block["kind"] == "DV" else 2 for block in BAO_BLOCKS) + 2

    def chi2_dvch(theta):
        H0, omega_m0, n, rd = theta
        allowed = (
            50.0 < H0 < 90.0
            and 0.05 < omega_m0 < 0.6
            and 0.0 < n < 1.0
            and 120.0 < rd < 170.0
        )
        if not allowed:
            return 1e50
        try:
            z_grid, e_grid = solve_dvch_grid(
                zmax=zmax,
                H0=H0,
                omega_m0=omega_m0,
                omega_r0=config.omega_r0,
                n=n,
                beta=config.beta,
            )
            return build_summary(
                "DVCH", theta, z_grid, e_grid, H0, rd, 4, n_data, config.bridge_pivot_z
            ).chi2_total
        except Exception:
            return 1e50

    def chi2_lcdm(theta):
        H0, omega_m0, rd = theta
        allowed = (
            50.0 < H0 < 90.0
            and 0.05 < omega_m0 < 0.6
            and 120.0 < rd < 170.0
        )
        if not allowed:
            return 1e50
        try:
            z_grid, e_grid = solve_lcdm_grid(
                zmax=zmax,
                omega_m0=omega_m0,
                omega_r0=config.omega_r0,
            )
            return build_summary(
                "LCDM", theta, z_grid, e_grid, H0, rd, 3, n_data, config.bridge_pivot_z
            ).chi2_total
        except Exception:
            return 1e50

    print("Ajustando DVCH con BAO + chronometers + anclas H0...")
    res_dvch = minimize(
        chi2_dvch,
        x0=np.array(config.dvch_init),
        method="Nelder-Mead",
        options={"maxiter": config.maxiter, "xatol": config.xatol, "fatol": config.fatol},
    )

    print("Ajustando LCDM con BAO + chronometers + anclas H0...")
    res_lcdm = minimize(
        chi2_lcdm,
        x0=np.array(config.lcdm_init),
        method="Nelder-Mead",
        options={"maxiter": config.maxiter, "xatol": config.xatol, "fatol": config.fatol},
    )

    z_dvch, e_dvch = solve_dvch_grid(
        zmax=zmax,
        H0=float(res_dvch.x[0]),
        omega_m0=float(res_dvch.x[1]),
        omega_r0=config.omega_r0,
        n=float(res_dvch.x[2]),
        beta=config.beta,
    )
    z_lcdm, e_lcdm = solve_lcdm_grid(
        zmax=zmax,
        omega_m0=float(res_lcdm.x[1]),
        omega_r0=config.omega_r0,
    )

    dvch_summary = build_summary(
        "DVCH",
        res_dvch.x,
        z_dvch,
        e_dvch,
        float(res_dvch.x[0]),
        float(res_dvch.x[3]),
        4,
        n_data,
        config.bridge_pivot_z,
    )

    lcdm_summary = build_summary(
        "LCDM",
        res_lcdm.x,
        z_lcdm,
        e_lcdm,
        float(res_lcdm.x[0]),
        float(res_lcdm.x[2]),
        3,
        n_data,
        config.bridge_pivot_z,
    )

    results = JointFitResults(
        assumptions={
            "datasets": [
                "DESI DR1 BAO compressed",
                "Cosmic chronometers",
                "Planck H0 anchor",
                "SH0ES H0 anchor",
            ],
            "omega_r0": config.omega_r0,
            "beta": config.beta,
            "optimizer": "Nelder-Mead",
            "bridge_quantity": "H(z)/(1+z)",
            "bridge_pivot_z": config.bridge_pivot_z,
            "planck_anchor": {"mean": PLANCK_H0, "sigma": PLANCK_SIGMA},
            "shoes_anchor": {"mean": SHOES_H0, "sigma": SHOES_SIGMA},
        },
        n_data=n_data,
        dvch=dvch_summary,
        lcdm=lcdm_summary,
        delta_chi2=float(dvch_summary.chi2_total - lcdm_summary.chi2_total),
        delta_aic=float(dvch_summary.aic - lcdm_summary.aic),
        delta_bic=float(dvch_summary.bic - lcdm_summary.bic),
    )

    return results, (z_dvch, e_dvch, z_lcdm, e_lcdm)


def save_svg_fallback(results, grids, filename="Fig_DVCH_union_realdata.svg"):
    z_dvch, e_dvch, z_lcdm, e_lcdm = grids
    bridge_dvch = bridge_h0_curve(z_dvch, e_dvch, results.dvch.best_fit[0])
    bridge_lcdm = bridge_h0_curve(z_lcdm, e_lcdm, results.lcdm.best_fit[0])

    width, height = 1100, 420
    margin_left, margin_bottom, margin_top = 70, 55, 30
    panel_gap = 40
    panel_width = (width - margin_left - 40 - panel_gap) / 2
    panel_height = height - margin_top - margin_bottom

    z_max = max(float(z_dvch.max()), 2.35)
    y_min, y_max = 62.5, 76.5
    bar_max = max(results.dvch.chi2_h0_pair, results.lcdm.chi2_h0_pair, 4.0) * 1.18

    def map_xy(z, y):
        xpix = margin_left + (z / z_max) * panel_width
        ypix = margin_top + panel_height * (1.0 - (y - y_min) / (y_max - y_min))
        return xpix, ypix

    def polyline(xs, ys):
        return " ".join(f"{x:.2f},{y:.2f}" for x, y in zip(xs, ys))

    dv_x, dv_y = zip(*(map_xy(float(z), float(y)) for z, y in zip(z_dvch, bridge_dvch)))
    lc_x, lc_y = zip(*(map_xy(float(z), float(y)) for z, y in zip(z_lcdm, bridge_lcdm)))

    panel2_left = margin_left + panel_width + panel_gap
    bar_w = 90
    centers = [panel2_left + 110, panel2_left + 260]

    def bar_top(value):
        return margin_top + panel_height * (1.0 - value / bar_max)

    svg = f'''<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">
  <rect width="100%" height="100%" fill="white"/>
  <text x="{width/2:.0f}" y="20" text-anchor="middle" font-size="18" font-family="Arial">Test de Unión DVCH con datos reales</text>

  <rect x="{margin_left}" y="{map_xy(0, PLANCK_H0 + PLANCK_SIGMA)[1]:.2f}" width="{panel_width}" height="{abs(map_xy(0, PLANCK_H0 - PLANCK_SIGMA)[1] - map_xy(0, PLANCK_H0 + PLANCK_SIGMA)[1]):.2f}" fill="#999999" opacity="0.25"/>
  <rect x="{margin_left}" y="{map_xy(0, SHOES_H0 + SHOES_SIGMA)[1]:.2f}" width="{panel_width}" height="{abs(map_xy(0, SHOES_H0 - SHOES_SIGMA)[1] - map_xy(0, SHOES_H0 + SHOES_SIGMA)[1]):.2f}" fill="#2e8b57" opacity="0.18"/>
  <line x1="{margin_left}" y1="{margin_top}" x2="{margin_left}" y2="{margin_top + panel_height}" stroke="black"/>
  <line x1="{margin_left}" y1="{margin_top + panel_height}" x2="{margin_left + panel_width}" y2="{margin_top + panel_height}" stroke="black"/>
  <polyline fill="none" stroke="#1f77b4" stroke-width="2.5" points="{polyline(dv_x, dv_y)}"/>
  <polyline fill="none" stroke="#d62728" stroke-width="2" stroke-dasharray="7,5" points="{polyline(lc_x, lc_y)}"/>
  <circle cx="{map_xy(results.assumptions['bridge_pivot_z'], results.dvch.bridge_highz)[0]:.2f}" cy="{map_xy(results.assumptions['bridge_pivot_z'], results.dvch.bridge_highz)[1]:.2f}" r="4" fill="#1f77b4"/>
  <circle cx="{map_xy(0.0, results.dvch.bridge_lowz)[0]:.2f}" cy="{map_xy(0.0, results.dvch.bridge_lowz)[1]:.2f}" r="4" fill="#1f77b4"/>
  <text x="{margin_left + panel_width/2:.2f}" y="{height - 12}" text-anchor="middle" font-size="13" font-family="Arial">Redshift z</text>
  <text x="20" y="{margin_top + panel_height/2:.2f}" text-anchor="middle" font-size="13" font-family="Arial" transform="rotate(-90 20,{margin_top + panel_height/2:.2f})">H(z)/(1+z) [km s^-1 Mpc^-1]</text>
  <text x="{margin_left + panel_width/2:.2f}" y="{margin_top - 8}" text-anchor="middle" font-size="14" font-family="Arial">Puente DVCH entre Planck y SH0ES</text>

  <line x1="{panel2_left}" y1="{margin_top}" x2="{panel2_left}" y2="{margin_top + panel_height}" stroke="black"/>
  <line x1="{panel2_left}" y1="{margin_top + panel_height}" x2="{panel2_left + panel_width}" y2="{margin_top + panel_height}" stroke="black"/>
  <line x1="{panel2_left}" y1="{bar_top(4.0):.2f}" x2="{panel2_left + panel_width}" y2="{bar_top(4.0):.2f}" stroke="black" stroke-dasharray="4,4"/>
  <rect x="{centers[0] - bar_w/2:.2f}" y="{bar_top(results.dvch.chi2_h0_pair):.2f}" width="{bar_w}" height="{margin_top + panel_height - bar_top(results.dvch.chi2_h0_pair):.2f}" fill="#1f77b4" opacity="0.85"/>
  <rect x="{centers[1] - bar_w/2:.2f}" y="{bar_top(results.lcdm.chi2_h0_pair):.2f}" width="{bar_w}" height="{margin_top + panel_height - bar_top(results.lcdm.chi2_h0_pair):.2f}" fill="#d62728" opacity="0.85"/>
  <text x="{centers[0]:.2f}" y="{bar_top(results.dvch.chi2_h0_pair) - 8:.2f}" text-anchor="middle" font-size="13" font-family="Arial">{results.dvch.sigma_pair:.2f}σ</text>
  <text x="{centers[1]:.2f}" y="{bar_top(results.lcdm.chi2_h0_pair) - 8:.2f}" text-anchor="middle" font-size="13" font-family="Arial">{results.lcdm.sigma_pair:.2f}σ</text>
  <text x="{centers[0]:.2f}" y="{margin_top + panel_height + 20}" text-anchor="middle" font-size="13" font-family="Arial">DVCH</text>
  <text x="{centers[1]:.2f}" y="{margin_top + panel_height + 20}" text-anchor="middle" font-size="13" font-family="Arial">ΛCDM</text>
  <text x="{panel2_left + panel_width/2:.2f}" y="{margin_top - 8}" text-anchor="middle" font-size="14" font-family="Arial">Consistencia simultánea</text>
  <text x="{panel2_left + panel_width/2:.2f}" y="{height - 12}" text-anchor="middle" font-size="13" font-family="Arial">Modelo</text>
  <text x="{panel2_left - 40:.2f}" y="{margin_top + panel_height/2:.2f}" text-anchor="middle" font-size="13" font-family="Arial" transform="rotate(-90 {panel2_left - 40:.2f},{margin_top + panel_height/2:.2f})">χ²_H0,pair</text>
</svg>'''

    with open(filename, "w", encoding="utf-8") as handle:
        handle.write(svg)


def make_union_plot(results, grids, filename="Fig_DVCH_union_realdata"):
    if not HAS_MATPLOTLIB:
        save_svg_fallback(results, grids, filename + ".svg")
        return [filename + ".svg"]

    z_dvch, e_dvch, z_lcdm, e_lcdm = grids
    dvch_H0 = results.dvch.best_fit[0]
    lcdm_H0 = results.lcdm.best_fit[0]

    bridge_dvch = bridge_h0_curve(z_dvch, e_dvch, dvch_H0)
    bridge_lcdm = bridge_h0_curve(z_lcdm, e_lcdm, lcdm_H0)

    plt.style.use("seaborn-v0_8-whitegrid")
    fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.8))

    axes[0].fill_between(
        z_dvch,
        PLANCK_H0 - PLANCK_SIGMA,
        PLANCK_H0 + PLANCK_SIGMA,
        color="gray",
        alpha=0.25,
        label=r"Planck $67.4\pm0.5$",
    )
    axes[0].fill_between(
        z_dvch,
        SHOES_H0 - SHOES_SIGMA,
        SHOES_H0 + SHOES_SIGMA,
        color="forestgreen",
        alpha=0.20,
        label=r"SH0ES $73.0\pm1.0$",
    )
    axes[0].plot(z_dvch, bridge_dvch, lw=2.4, color="tab:blue", label="DVCH")
    axes[0].plot(z_lcdm, bridge_lcdm, lw=2.0, ls="--", color="tab:red", label=r"$\Lambda$CDM")
    axes[0].scatter(
        [results.assumptions["bridge_pivot_z"], 0.0],
        [results.dvch.bridge_highz, results.dvch.bridge_lowz],
        color="tab:blue",
        s=28,
        zorder=5,
    )
    axes[0].set_xlim(0.0, max(2.35, z_dvch.max()))
    axes[0].set_ylim(62.5, 76.5)
    axes[0].set_xlabel("Redshift z")
    axes[0].set_ylabel(r"$H(z)/(1+z)$ [km s$^{-1}$ Mpc$^{-1}$]")
    axes[0].set_title("Puente DVCH entre Planck y SH0ES")
    axes[0].legend(fontsize=8)

    labels = ["DVCH", r"$\Lambda$CDM"]
    chi2_pair = [results.dvch.chi2_h0_pair, results.lcdm.chi2_h0_pair]
    sigma_pair = [results.dvch.sigma_pair, results.lcdm.sigma_pair]
    x = np.arange(len(labels))
    bars = axes[1].bar(
        x,
        chi2_pair,
        width=0.45,
        color=["tab:blue", "tab:red"],
        alpha=0.85,
    )
    axes[1].axhline(4.0, color="k", ls=":", lw=1.0, label=r"Umbral $2\sigma$ ($\chi^2=4$)")
    axes[1].set_xticks(x, labels)
    axes[1].set_ylabel(r"$\chi^2_{H_0,\mathrm{pair}}$")
    axes[1].set_title("Consistencia simultánea")

    for bar, sigma in zip(bars, sigma_pair):
        axes[1].text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.15,
            f"{sigma:.2f}" + r"$\sigma$",
            ha="center",
            va="bottom",
            fontsize=9,
        )

    axes[1].legend(fontsize=8)

    fig.suptitle("Test de Unión DVCH con datos reales", y=1.02)
    fig.tight_layout()
    png_path = filename + ".png"
    pdf_path = filename + ".pdf"
    fig.savefig(png_path, dpi=300, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    plt.close(fig)
    return [png_path, pdf_path]


def save_results_json(results, filename="joint_fit_dvch_realdata_selfcontained.json"):
    with open(filename, "w", encoding="utf-8") as handle:
        json.dump(asdict(results), handle, indent=2)
    return filename


def print_summary(results):
    print("\n================ RESULTADOS ================\n")
    print("Datos reales usados: DESI DR1 BAO + cosmic chronometers + anclas Planck/SH0ES")
    print()

    for summary in (results.dvch, results.lcdm):
        print(summary.model)
        print("best fit =", np.array(summary.best_fit))
        print(
            f"chi2_total = {summary.chi2_total:.6f} | "
            f"BAO = {summary.chi2_bao:.3f}, "
            f"CC = {summary.chi2_cc:.3f}, "
            f"H0-pair = {summary.chi2_h0_pair:.3f}"
        )
        print(
            f"bridge(z={results.assumptions['bridge_pivot_z']:.2f}) = {summary.bridge_highz:.3f}, "
            f"bridge(z=0) = {summary.bridge_lowz:.3f}, "
            f"tension = {summary.sigma_pair:.3f} sigma"
        )
        print(f"AIC = {summary.aic:.6f} | BIC = {summary.bic:.6f}")
        print()

    print(f"Delta chi2 (DVCH - LCDM) = {results.delta_chi2:.6f}")
    print(f"Delta AIC  (DVCH - LCDM) = {results.delta_aic:.6f}")
    print(f"Delta BIC  (DVCH - LCDM) = {results.delta_bic:.6f}")
    print("\n==========================================\n")


def main():
    results, grids = run_joint_fit()
    print_summary(results)
    json_path = save_results_json(results)
    figure_paths = make_union_plot(results, grids)

    print("Archivos generados:")
    for path in figure_paths:
        print(" -", path)
    print(" -", json_path)

    if not HAS_MATPLOTLIB:
        print("Nota: `matplotlib` no estaba disponible; se generó SVG autosuficiente.")


if __name__ == "__main__":
    main()

Ajustando DVCH con BAO + chronometers + anclas H0...
Ajustando LCDM con BAO + chronometers + anclas H0...

================ RESULTADOS ================

Datos reales usados: DESI DR1 BAO + cosmic chronometers + anclas Planck/SH0ES

DVCH
best fit = [7.11797646e+01 2.53393483e-01 2.60979428e-15 1.48093291e+02]
chi2_total = 39.392141 | BAO = 22.145, CC = 12.558, H0-pair = 4.689
bridge(z=2.33) = 67.986, bridge(z=0) = 71.180, tension = 2.165 sigma
AIC = 47.392141 | BIC = 54.706707

LCDM
best fit = [ 71.17933354   0.25344921 148.09077047]
chi2_total = 39.394398 | BAO = 22.147, CC = 12.557, H0-pair = 4.691
bridge(z=2.33) = 67.987, bridge(z=0) = 71.179, tension = 2.166 sigma
AIC = 45.394398 | BIC = 50.880322

Delta chi2 (DVCH - LCDM) = -0.002256
Delta AIC  (DVCH - LCDM) = 1.997744
Delta BIC  (DVCH - LCDM) = 3.826385


Archivos generados:
 - Fig_DVCH_union_realdata.png
 - Fig_DVCH_union_realdata.pdf
 - joint_fit_dvch_realdata_selfcontained.json
